[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ai-forever/FRIDA-Decisions/blob/grok/examples-chat_filter/examples/chat_filter/chat_filter.ipynb)

# Filter a group chat

Each message is a handful of yes/no questions: insult, spam, spoiler, unsolicited tip, question. The action is a rule in code, in a fixed order, so a message that is both a question and spam is hidden rather than highlighted. Messages in flight at the same time share one `judge_batch` inside `AsyncJudge`.

`noul` returns a probability and no confidence. The cutoffs are the policy.

Pattern: [Twitch chat filter](https://docs.typesafe.ai/cookbooks/twitch_chat_filter), retargeted at a Russian game chat. The insults are mild and synthetic.


In [1]:
import sys
!{sys.executable} -m pip install -q "frida-decisions[torch] @ git+https://github.com/ai-forever/FRIDA-Decisions@v0.4.0"


In [2]:
import json, os
from pathlib import Path
import torch
import frida_decisions

MODEL = os.environ.get("FD_MODEL", "ai-forever/FRIDA-Decisions")   # repo id or a local folder
device = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cpu":
    torch.set_num_threads(min(4, os.cpu_count() or 1))
print(f"frida-decisions {frida_decisions.__version__}, device {device}")


frida-decisions 0.4.0, device cpu


In [3]:
import asyncio
import time

# First match wins. A question that is also spam is hidden, not highlighted.
# Question-ness is weak on its own, so a highlight also requires a question mark.
HIDE_INSULT = 0.70
HIDE_SPAM = 0.60
BLUR_SPOILER = 0.75
COLLAPSE_TIP = 0.70
HIGHLIGHT_QUESTION = 0.35


def request_for(text: str) -> dict:
    return {
        "state": text,
        "questions": {
            "insult": {"type": "noul", "instructions": "В сообщении оскорбляют человека?"},
            "spam": {"type": "noul", "instructions": "Это реклама или навязчивый призыв купить?"},
            "spoiler": {"type": "noul", "instructions": "Сообщение заранее раскрывает сюжет или развязку?"},
            "tip": {"type": "noul", "instructions": "Это непрошеный совет, как играть или что настроить?"},
            "question": {"type": "noul", "instructions": "Автор задаёт вопрос другим участникам?"},
        },
    }


def moderate(text: str, answers: dict) -> str:
    if answers["insult"]["noul"] >= HIDE_INSULT or answers["spam"]["noul"] >= HIDE_SPAM:
        return "hide"
    if answers["spoiler"]["noul"] >= BLUR_SPOILER:
        return "blur"
    if answers["tip"]["noul"] >= COLLAPSE_TIP:
        return "collapse"
    if "?" in text and answers["question"]["noul"] >= HIGHLIGHT_QUESTION:
        return "highlight"
    return "show"


## Several messages at once

`asyncio.gather` awaits every message together. `AsyncJudge` scores that group in one `judge_batch` on the worker thread, so the event loop stays free while the model runs.


In [4]:
async def run(judge, rows: list[dict]) -> None:
    started = time.perf_counter()
    responses = await asyncio.gather(*(judge.judge(request_for(row["text"])) for row in rows))
    elapsed = time.perf_counter() - started
    hits = 0
    for row, response in zip(rows, responses):
        action = moderate(row["text"], response["answers"])
        hits += action == row["action"]
        probs = " ".join(f"{name}={response['answers'][name]['noul']:.2f}"
                         for name in ("insult", "spam", "spoiler", "tip", "question"))
        print(f"{'ok' if action == row['action'] else 'miss':4} got {action:10} gold {row['action']:10} {probs}")
        print(f"     {row['text']}")
    print(f"\n{hits}/{len(rows)} actions match, {len(rows)} messages in {elapsed:.2f}s")


In [5]:
def load_rows():
    import urllib.request
    path = Path("data.json")
    if not path.exists():
        url = ("https://raw.githubusercontent.com/ai-forever/FRIDA-Decisions/"
               "grok/examples-chat-filter/examples/chat_filter/data.json")
        path.write_bytes(urllib.request.urlopen(url, timeout=60).read())
    return json.loads(path.read_text(encoding="utf-8"))

from frida_decisions import AsyncJudge

async with AsyncJudge.from_pretrained(MODEL, device=device) as judge:
    await run(judge, load_rows())


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/219 [00:00<?, ?it/s]

ok   got highlight  gold highlight  insult=0.08 spam=0.10 spoiler=0.41 tip=0.26 question=0.38
     Подскажите, пожалуйста, где крафтится верстак?
ok   got hide       gold hide       insult=0.30 spam=0.79 spoiler=0.65 tip=0.49 question=0.70
     Купите золото дёшево, пишите в личные сообщения, лучшая цена на сервере!!!
ok   got hide       gold hide       insult=0.92 spam=0.08 spoiler=0.51 tip=0.50 question=0.72
     Ты тупица и играешь отвратительно.
ok   got blur       gold blur       insult=0.25 spam=0.23 spoiler=0.81 tip=0.50 question=0.78
     Осторожно, спойлер: босс умирает с одного удара в глаз.
ok   got collapse   gold collapse   insult=0.11 spam=0.17 spoiler=0.53 tip=0.81 question=0.77
     Поставь чувствительность мыши на 800, так лучше целятся.
ok   got show       gold show       insult=0.01 spam=0.03 spoiler=0.29 tip=0.29 question=0.47
     Спасибо за игру, до завтра.
ok   got blur       gold blur       insult=0.19 spam=0.22 spoiler=0.95 tip=0.53 question=0.83
     Спойлер п